# Five AI risk categories: equal-weight portfolio Granger tests

This notebook tests the five hourly category indices separately against the fixed 20-stock equal-weight portfolio:

- `loss_of_human_control`
- `malicious_use`
- `job_displacement`
- `discrimination`
- `privacy_violation`

Each category is entered in its own model, so the test asks whether that category has predictive content without claiming an effect independent of the other categories. The selected return lag order is inherited from a fresh BIC selection on the return-only common sample. Risk lag orders are $p=1,2,3$.

Within each category, all three $p$ values use that category's same $p=3$-complete sample. Category samples may differ slightly because warm-up missingness differs by index. Trading-date clustered F tests are primary, HAC(5 observations) is a sensitivity check, and the confirmatory BH correction covers every estimable category-by-$p$ test. A category with no sample variation is reported as **not estimable**, never as a non-significant zero effect.

In [1]:
from pathlib import Path
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import scipy
import statsmodels
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

ROOT = Path.cwd().resolve()
if not (ROOT / "download_hourly_returns.py").exists():
    raise RuntimeError("Run this notebook from the project root directory.")

REGULAR_PATH = ROOT / "data" / "hourly_10_16" / "hourly_returns_long.csv"
BRIDGE_PATH = ROOT / "data" / "premarket_bridge_07_10" / "hourly_returns_long.csv"
RISK_PATH = ROOT / "data" / "hourly_ai_risk_index" / "x" / "hourly_ai_risk_index.csv"

for path in (REGULAR_PATH, BRIDGE_PATH, RISK_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

environment = pd.DataFrame(
    {
        "component": ["Python", "pandas", "NumPy", "SciPy", "statsmodels", "platform"],
        "version": [
            platform.python_version(),
            pd.__version__,
            np.__version__,
            scipy.__version__,
            statsmodels.__version__,
            platform.platform(),
        ],
    }
)
display(environment)
print("Interpreter:", sys.executable)
print("Project root:", ROOT)

,component,version
0,Python,3.14.7
1,pandas,3.0.6
2,NumPy,2.5.3
3,SciPy,1.18.1
4,statsmodels,0.15.0
5,platform,Windows-11-10.0.26200-SP0


Interpreter: C:\Users\Jerry Cai\Desktop\NLP_Assignment4_AI_risk_and_AI_development\.venv\Scripts\python.exe
Project root: C:\Users\Jerry Cai\Desktop\NLP_Assignment4_AI_risk_and_AI_development


## 1. Load, validate and align the five category indices

In [2]:
def parse_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().eq("true")


def load_return_file(path: Path, expected_mode: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {
        "ticker", "trading_date", "bar_start_utc", "bar_end_utc",
        "duration_minutes", "window_complete", "ohlc_valid", "nonpositive_price",
        "split_event", "log_ret_oc", "return_requires_review",
    }
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    frame["bar_start_utc"] = pd.to_datetime(frame["bar_start_utc"], utc=True)
    frame["bar_end_utc"] = pd.to_datetime(frame["bar_end_utc"], utc=True)
    frame["window_complete"] = parse_bool(frame["window_complete"])
    frame["ohlc_valid"] = parse_bool(frame["ohlc_valid"])
    frame["nonpositive_price"] = parse_bool(frame["nonpositive_price"])
    frame["split_event"] = parse_bool(frame["split_event"])
    frame["return_requires_review"] = parse_bool(frame["return_requires_review"])
    frame["duration_minutes"] = pd.to_numeric(frame["duration_minutes"], errors="coerce")
    frame["log_ret_oc"] = pd.to_numeric(frame["log_ret_oc"], errors="coerce")
    if "window_mode" in frame.columns and set(frame["window_mode"].dropna().unique()) != {expected_mode}:
        raise ValueError(f"Unexpected window mode in {path}")
    if frame.duplicated(["ticker", "bar_start_utc"]).any():
        raise ValueError(f"Duplicate ticker/start timestamps in {path}")
    return frame


regular = load_return_file(REGULAR_PATH, "clock_10_16")
bridge = load_return_file(BRIDGE_PATH, "bridge_07_10")
risk = pd.read_csv(RISK_PATH)

RISK_CATEGORIES = (
    "loss_of_human_control",
    "malicious_use",
    "job_displacement",
    "discrimination",
    "privacy_violation",
)
CATEGORY_COLUMNS = {category: f"AI_Risk_{category}" for category in RISK_CATEGORIES}
CATEGORY_STATUS_COLUMNS = {category: f"AI_Risk_{category}_status" for category in RISK_CATEGORIES}

risk_required = {
    "hour", "AI_Risk_total", "coverage_status", "coverage_complete",
    "is_backtest_eligible_observed_sample", "is_backtest_eligible_strict",
    *CATEGORY_COLUMNS.values(),
    *CATEGORY_STATUS_COLUMNS.values(),
}
missing = risk_required.difference(risk.columns)
if missing:
    raise ValueError(f"Risk index is missing columns: {sorted(missing)}")

risk["hour"] = pd.to_datetime(risk["hour"], utc=True)
risk["AI_Risk_total"] = pd.to_numeric(risk["AI_Risk_total"], errors="coerce")
for column in CATEGORY_COLUMNS.values():
    risk[column] = pd.to_numeric(risk[column], errors="coerce")
risk["observed_eligible"] = parse_bool(risk["is_backtest_eligible_observed_sample"])
risk["strict_eligible"] = parse_bool(risk["is_backtest_eligible_strict"])
allowed_category_statuses = {"computed", "no_relevant_documents", "confirmed_no_news"}
for category in RISK_CATEGORIES:
    value_column = CATEGORY_COLUMNS[category]
    status_column = CATEGORY_STATUS_COLUMNS[category]
    numeric_observed = risk["observed_eligible"] & risk[value_column].notna()
    bad_status = numeric_observed & ~risk[status_column].isin(allowed_category_statuses)
    if bad_status.any():
        raise ValueError(
            f"{category} has numeric observed-eligible values with invalid status: "
            f"{sorted(risk.loc[bad_status, status_column].astype(str).unique())}"
        )
if risk["hour"].duplicated().any():
    raise ValueError("Risk index contains duplicate hourly timestamps")
if len(risk) > 1 and not risk["hour"].sort_values().diff().dropna().eq(pd.Timedelta(hours=1)).all():
    raise ValueError("Risk index is not a continuous hourly grid")

regular_hours = set(regular["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour.unique())
bridge_hours = set(bridge["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour.unique())
assert regular_hours == set(range(10, 16)), regular_hours
assert bridge_hours == {7, 8, 9}, bridge_hours
assert regular["duration_minutes"].eq(60).all()

targets = regular.loc[
    regular["window_complete"]
    & regular["ohlc_valid"]
    & ~regular["nonpositive_price"]
    & ~regular["split_event"]
    & ~regular["return_requires_review"]
    & regular["log_ret_oc"].notna()
].copy()
targets = targets.rename(columns={"log_ret_oc": "target_return"})
targets["target_start_ny"] = targets["bar_start_utc"].dt.tz_convert("America/New_York")
targets["target_end_ny"] = targets["bar_end_utc"].dt.tz_convert("America/New_York")
targets["target_hour"] = targets["target_start_ny"].dt.hour.astype(str).str.zfill(2) + ":00"
targets["trading_date"] = targets["trading_date"].astype(str)
targets["weekday"] = pd.to_datetime(targets["trading_date"]).dt.day_name()

return_grid = pd.concat(
    [
        bridge[["ticker", "trading_date", "bar_start_utc", "window_complete", "ohlc_valid", "nonpositive_price", "split_event", "return_requires_review", "log_ret_oc"]],
        regular[["ticker", "trading_date", "bar_start_utc", "window_complete", "ohlc_valid", "nonpositive_price", "split_event", "return_requires_review", "log_ret_oc"]],
    ],
    ignore_index=True,
)
usable_return = (
    return_grid["window_complete"]
    & return_grid["ohlc_valid"]
    & ~return_grid["nonpositive_price"]
    & ~return_grid["split_event"]
    & ~return_grid["return_requires_review"]
)
return_grid.loc[~usable_return, "log_ret_oc"] = np.nan
return_grid["trading_date"] = return_grid["trading_date"].astype(str)
if return_grid.duplicated(["ticker", "bar_start_utc"]).any():
    raise ValueError("Combined bridge/regular return grid has duplicate keys")

return_lookup = return_grid.set_index(["ticker", "bar_start_utc"])["log_ret_oc"]
date_lookup = return_grid.set_index(["ticker", "bar_start_utc"])["trading_date"]
risk_value_lookup = risk.set_index("hour")["AI_Risk_total"]
risk_eligible_lookup = risk.set_index("hour")["observed_eligible"]
risk_category_lookups = {
    category: risk.set_index("hour")[column]
    for category, column in CATEGORY_COLUMNS.items()
}

for lag in range(1, 4):
    return_key_time = targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    return_keys = pd.MultiIndex.from_arrays([targets["ticker"], return_key_time])
    targets[f"return_lag_{lag}"] = return_lookup.reindex(return_keys).to_numpy()
    lag_dates = date_lookup.reindex(return_keys).to_numpy()
    same_day = lag_dates == targets["trading_date"].to_numpy()
    targets.loc[~same_day, f"return_lag_{lag}"] = np.nan

    risk_hour = targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    targets[f"risk_hour_lag_{lag}"] = risk_hour
    targets[f"risk_lag_{lag}"] = risk_value_lookup.reindex(risk_hour).to_numpy()
    eligible = risk_eligible_lookup.reindex(risk_hour)
    targets[f"risk_eligible_lag_{lag}"] = eligible.fillna(False).astype(bool).to_numpy()
    for category, lookup in risk_category_lookups.items():
        targets[f"risk_{category}_lag_{lag}"] = lookup.reindex(risk_hour).to_numpy()

targets = targets.sort_values(["bar_start_utc", "ticker"]).reset_index(drop=True)

assert (targets["risk_hour_lag_1"] + pd.Timedelta(hours=1) == targets["bar_start_utc"]).all()
assert not targets.duplicated(["ticker", "bar_start_utc"]).any()

audit = pd.DataFrame(
    [
        ("regular target rows", len(targets)),
        ("tickers", targets["ticker"].nunique()),
        ("regular trading dates", targets["trading_date"].nunique()),
        ("bridge expected rows", len(bridge)),
        ("bridge complete rows", int(bridge["window_complete"].sum())),
        ("risk hourly rows", len(risk)),
        ("risk observed-sample eligible rows", int(risk["observed_eligible"].sum())),
        ("risk strict-coverage eligible rows", int(risk["strict_eligible"].sum())),
        ("risk queries at result cap", int(pd.to_numeric(risk.get("queries_at_result_cap"), errors="coerce").max())),
    ],
    columns=["check", "value"],
)
display(audit)

example_row = targets.loc[
    targets["ticker"].eq("MSFT")
    & targets["target_start_ny"].dt.hour.eq(10)
    & targets["risk_lag_3"].notna()
].iloc[0]
alignment_example = pd.DataFrame(
    {
        "role": ["target return", "lag 1 risk/return", "lag 2 risk/return", "lag 3 risk/return"],
        "window_start_NY": [
            example_row["target_start_ny"],
            example_row["target_start_ny"] - pd.Timedelta(hours=1),
            example_row["target_start_ny"] - pd.Timedelta(hours=2),
            example_row["target_start_ny"] - pd.Timedelta(hours=3),
        ],
        "window_end_NY": [
            example_row["target_end_ny"],
            example_row["target_start_ny"],
            example_row["target_start_ny"] - pd.Timedelta(hours=1),
            example_row["target_start_ny"] - pd.Timedelta(hours=2),
        ],
    }
)
display(Markdown("**Exact no-look-ahead example (MSFT):**"))
display(alignment_example)

,check,value
0,regular target rows,2760
1,tickers,20
2,regular trading dates,23
3,bridge expected rows,1380
4,bridge complete rows,887
5,risk hourly rows,723
6,risk observed-sample eligible rows,622
7,risk strict-coverage eligible rows,0
8,risk queries at result cap,33


**Exact no-look-ahead example (MSFT):**

,role,window_start_NY,window_end_NY
0,target return,2026-09-02 10:00:00-04:00,2026-09-02 11:00:00-04:00
1,lag 1 risk/return,2026-09-02 09:00:00-04:00,2026-09-02 10:00:00-04:00
2,lag 2 risk/return,2026-09-02 08:00:00-04:00,2026-09-02 09:00:00-04:00
3,lag 3 risk/return,2026-09-02 07:00:00-04:00,2026-09-02 08:00:00-04:00


In [3]:
category_hour_rows = []
for category, column in CATEGORY_COLUMNS.items():
    values = risk.loc[risk["observed_eligible"], column]
    category_hour_rows.append(
        {
            "risk_category": category,
            "eligible_hours": int(risk["observed_eligible"].sum()),
            "nonmissing_hours": int(values.notna().sum()),
            "missing_hours": int(values.isna().sum()),
            "nonzero_hours": int(values.fillna(0).ne(0).sum()),
            "mean": float(values.mean()),
            "standard_deviation": float(values.std(ddof=1)),
            "minimum": float(values.min()),
            "maximum": float(values.max()),
            "unique_nonmissing_values": int(values.nunique(dropna=True)),
        }
    )

category_hour_summary = pd.DataFrame(category_hour_rows)
display(Markdown("### 五类小时风险指标的可用性"))
display(
    category_hour_summary.style.format(
        {
            "mean": "{:.6f}",
            "standard_deviation": "{:.6f}",
            "minimum": "{:.6f}",
            "maximum": "{:.6f}",
        }
    ).hide(axis="index")
)

category_correlation = risk.loc[
    risk["observed_eligible"], list(CATEGORY_COLUMNS.values())
].corr()
category_correlation.index = RISK_CATEGORIES
category_correlation.columns = RISK_CATEGORIES
display(Markdown("### 五类风险指标相关系数（observed eligible hours）"))
display(category_correlation.style.format("{:.3f}"))

### 五类小时风险指标的可用性

risk_category,eligible_hours,nonmissing_hours,missing_hours,nonzero_hours,mean,standard_deviation,minimum,maximum,unique_nonmissing_values
loss_of_human_control,622,618,4,73,0.228345,0.942102,-4.575470,8.558760,74
malicious_use,622,616,6,70,0.392712,1.508288,-2.181184,11.958268,71
job_displacement,622,614,8,58,0.303981,1.540163,-7.118610,14.267419,58
discrimination,622,617,5,7,0.031958,1.120235,-16.822032,14.671401,8
privacy_violation,622,615,7,51,0.051397,1.399919,-6.366318,19.317177,52


### 五类风险指标相关系数（observed eligible hours）

,loss_of_human_control,malicious_use,job_displacement,discrimination,privacy_violation
loss_of_human_control,1.000,0.189,0.006,-0.007,0.020
malicious_use,0.189,1.000,0.026,0.043,0.120
job_displacement,0.006,0.026,1.000,0.211,0.027
discrimination,-0.007,0.043,0.211,1.000,0.005
privacy_violation,0.020,0.120,0.027,0.005,1.000


## 2. Construct the fixed-composition 20-stock equal-weight portfolio

In [4]:
REQUESTED_TICKERS = (
    "MSFT", "GOOGL", "META", "AMZN", "TSLA", "ORCL", "CRM", "NOW", "ADBE", "WDAY",
    "INTU", "PATH", "AAPL", "PLTR", "UBER", "SNOW", "SHOP", "RBLX", "SNAP", "PINS",
)
EXPECTED_TICKERS = tuple(sorted(regular["ticker"].unique()))
if set(EXPECTED_TICKERS) != set(REQUESTED_TICKERS):
    missing = sorted(set(REQUESTED_TICKERS) - set(EXPECTED_TICKERS))
    extra = sorted(set(EXPECTED_TICKERS) - set(REQUESTED_TICKERS))
    raise ValueError(f"Ticker pool mismatch; missing={missing}, extra={extra}")


def build_equal_weight_hours(frame: pd.DataFrame, source_role: str) -> pd.DataFrame:
    work = frame.copy()
    work["ret_oc"] = pd.to_numeric(work["ret_oc"], errors="coerce")
    work["usable_component"] = (
        work["window_complete"]
        & work["ohlc_valid"]
        & ~work["nonpositive_price"]
        & ~work["split_event"]
        & ~work["return_requires_review"]
        & work["ret_oc"].notna()
        & work["duration_minutes"].eq(60)
    )
    rows = []
    keys = ["trading_date", "bar_start_utc", "bar_end_utc"]
    for (trading_date, bar_start, bar_end), group in work.groupby(keys, sort=True):
        usable = group.loc[group["usable_component"]].copy()
        usable_tickers = set(usable["ticker"])
        complete = usable_tickers == set(EXPECTED_TICKERS) and len(usable) == len(EXPECTED_TICKERS)
        simple_return = float(usable["ret_oc"].mean()) if complete else np.nan
        if complete and simple_return <= -1:
            raise ValueError(f"Invalid equal-weight return at {bar_start}")
        rows.append(
            {
                "source_role": source_role,
                "trading_date": str(trading_date),
                "bar_start_utc": bar_start,
                "bar_end_utc": bar_end,
                "n_valid_constituents": int(usable["ticker"].nunique()),
                "all_20_constituents_valid": bool(complete),
                "portfolio_ret_oc": simple_return,
                "portfolio_log_ret_oc": float(np.log1p(simple_return)) if complete else np.nan,
            }
        )
    result = pd.DataFrame(rows).sort_values("bar_start_utc").reset_index(drop=True)
    if result["bar_start_utc"].duplicated().any():
        raise ValueError(f"Duplicate portfolio timestamps for {source_role}")
    return result


regular_portfolio = build_equal_weight_hours(regular, "regular_target")
bridge_portfolio = build_equal_weight_hours(bridge, "bridge_lag_only")
if not regular_portfolio["all_20_constituents_valid"].all():
    raise ValueError("At least one regular portfolio target lacks a valid constituent")

portfolio_grid = pd.concat([bridge_portfolio, regular_portfolio], ignore_index=True)
portfolio_grid = portfolio_grid.sort_values("bar_start_utc").reset_index(drop=True)
if portfolio_grid["bar_start_utc"].duplicated().any():
    raise ValueError("Combined portfolio grid contains duplicate timestamps")

portfolio_targets = regular_portfolio.rename(columns={"portfolio_log_ret_oc": "target_return"}).copy()
portfolio_targets["target_start_ny"] = portfolio_targets["bar_start_utc"].dt.tz_convert("America/New_York")
portfolio_targets["target_end_ny"] = portfolio_targets["bar_end_utc"].dt.tz_convert("America/New_York")
portfolio_targets["target_hour"] = portfolio_targets["target_start_ny"].dt.hour.astype(str).str.zfill(2) + ":00"
portfolio_targets["weekday"] = pd.to_datetime(portfolio_targets["trading_date"]).dt.day_name()

portfolio_return_lookup = portfolio_grid.set_index("bar_start_utc")["portfolio_log_ret_oc"]
portfolio_date_lookup = portfolio_grid.set_index("bar_start_utc")["trading_date"]
for lag in range(1, 4):
    lag_start = portfolio_targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    portfolio_targets[f"return_lag_{lag}"] = portfolio_return_lookup.reindex(lag_start).to_numpy()
    lag_dates = portfolio_date_lookup.reindex(lag_start).to_numpy()
    same_day = lag_dates == portfolio_targets["trading_date"].to_numpy()
    portfolio_targets.loc[~same_day, f"return_lag_{lag}"] = np.nan

    risk_hour = portfolio_targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    portfolio_targets[f"risk_hour_lag_{lag}"] = risk_hour
    portfolio_targets[f"risk_lag_{lag}"] = risk_value_lookup.reindex(risk_hour).to_numpy()
    for category, lookup in risk_category_lookups.items():
        portfolio_targets[f"risk_{category}_lag_{lag}"] = lookup.reindex(risk_hour).to_numpy()
    eligible = risk_eligible_lookup.reindex(risk_hour)
    portfolio_targets[f"risk_eligible_lag_{lag}"] = eligible.fillna(False).astype(bool).to_numpy()

assert (portfolio_targets["risk_hour_lag_1"] + pd.Timedelta(hours=1) == portfolio_targets["bar_start_utc"]).all()

portfolio_audit = pd.DataFrame(
    {
        "check": [
            "constituents", "regular target hours", "regular hours with all 20",
            "bridge hours", "bridge hours with all 20", "regular target dates",
        ],
        "value": [
            len(EXPECTED_TICKERS),
            len(regular_portfolio),
            int(regular_portfolio["all_20_constituents_valid"].sum()),
            len(bridge_portfolio),
            int(bridge_portfolio["all_20_constituents_valid"].sum()),
            regular_portfolio["trading_date"].nunique(),
        ],
    }
)
display(portfolio_audit)

bridge_constituent_counts = (
    bridge_portfolio.groupby(
        bridge_portfolio["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour
    )["n_valid_constituents"]
    .agg(["min", "median", "max"])
    .rename_axis("bridge_start_hour_NY")
)
display(Markdown("### Valid constituent counts in bridge hours"))
display(bridge_constituent_counts)

,check,value
0,constituents,20
1,regular target hours,138
2,regular hours with all 20,138
3,bridge hours,69
4,bridge hours with all 20,0
5,regular target dates,23


### Valid constituent counts in bridge hours

,min,median,max
bridge_start_hour_NY,,,
7,10,14,17
8,11,14,19
9,9,10,11


## 3. Select the portfolio return lag order by BIC

In [5]:
def portfolio_fixed_effects(frame: pd.DataFrame) -> pd.DataFrame:
    categories = frame[["target_hour", "weekday"]].astype(str)
    return pd.get_dummies(
        categories,
        columns=["target_hour", "weekday"],
        prefix=["hour", "weekday"],
        drop_first=True,
        dtype=float,
    )


def portfolio_design(frame: pd.DataFrame, q: int, category: str | None = None, p: int = 0) -> pd.DataFrame:
    matrix = pd.DataFrame({"const": np.ones(len(frame), dtype=float)}, index=frame.index)
    for lag in range(1, q + 1):
        matrix[f"return_lag_{lag}"] = frame[f"return_lag_{lag}"].astype(float)
    if category is not None:
        for lag in range(1, p + 1):
            matrix[f"risk_{category}_lag_{lag}"] = frame[f"risk_{category}_lag_{lag}"].astype(float)
    return pd.concat([matrix, portfolio_fixed_effects(frame)], axis=1).astype(float)


bic_mask = portfolio_targets["target_return"].notna()
for lag in range(1, 4):
    bic_mask &= portfolio_targets[f"return_lag_{lag}"].notna()
portfolio_bic_sample = portfolio_targets.loc[bic_mask].sort_values("bar_start_utc").copy()

bic_rows = []
for q in (1, 2, 3):
    X = portfolio_design(portfolio_bic_sample, q=q, p=0)
    if np.linalg.matrix_rank(X.to_numpy()) != X.shape[1]:
        raise ValueError(f"Rank-deficient portfolio BIC design for q={q}")
    fit = sm.OLS(portfolio_bic_sample["target_return"].astype(float), X, missing="raise").fit()
    bic_rows.append(
        {
            "q_return_lags": q,
            "n_obs_common_sample": int(fit.nobs),
            "n_parameters": int(fit.df_model + 1),
            "BIC": float(fit.bic),
            "adjusted_R2": float(fit.rsquared_adj),
        }
    )

portfolio_bic_results = pd.DataFrame(bic_rows).sort_values("q_return_lags").reset_index(drop=True)
portfolio_bic_results["delta_BIC"] = portfolio_bic_results["BIC"] - portfolio_bic_results["BIC"].min()
Q_SELECTED = int(
    portfolio_bic_results.sort_values(["BIC", "q_return_lags"]).iloc[0]["q_return_lags"]
)
portfolio_bic_results["selected"] = portfolio_bic_results["q_return_lags"].eq(Q_SELECTED)

display(
    portfolio_bic_results.style.format(
        {"BIC": "{:.3f}", "delta_BIC": "{:.3f}", "adjusted_R2": "{:.6f}"}
    ).hide(axis="index")
)
display(Markdown(f"**Selected portfolio return lag order: q={Q_SELECTED}.**"))
near_ties = portfolio_bic_results.loc[
    (portfolio_bic_results["delta_BIC"] > 0) & (portfolio_bic_results["delta_BIC"] < 2)
]
if not near_ties.empty:
    labels = ", ".join(
        f"q={int(row.q_return_lags)} (ΔBIC={row.delta_BIC:.3f})"
        for row in near_ties.itertuples()
    )
    display(Markdown(f"**Near-tied alternatives: {labels}.**"))

q_return_lags,n_obs_common_sample,n_parameters,BIC,adjusted_R2,delta_BIC,selected
1,69,8,-601.231,-0.054254,2.411,False
2,69,9,-597.130,-0.069763,6.512,False
3,69,10,-603.642,0.068997,0.000,True


**Selected portfolio return lag order: q=3.**

## 4. Category-specific Granger tests

For category $c$, the unrestricted model is

$$r^{EW}_t=\alpha+\sum_{j=1}^{q}\phi_jr^{EW}_{t-j}+\sum_{j=1}^{p}\beta_{c,j}Risk^{c}_{t-j}+\text{hour FE}+\text{weekday FE}+\epsilon_t.$$

The category-specific null is $H_0:\beta_{c,1}=\cdots=\beta_{c,p}=0$. Risk from $[t-1,t)$ or earlier predicts return during $[t,t+1)$; no contemporaneous risk is used.

In [6]:
CATEGORY_SD = {
    category: float(risk.loc[risk["observed_eligible"], column].std(ddof=1))
    for category, column in CATEGORY_COLUMNS.items()
}
if any(not np.isfinite(value) or value <= 0 for value in CATEGORY_SD.values()):
    raise ValueError(f"At least one category has no full-period variation: {CATEGORY_SD}")

# Each category uses its own p=3-complete sample; p=1,2,3 remain directly comparable within category.
category_base_mask = portfolio_targets["target_return"].notna()
for lag in range(1, Q_SELECTED + 1):
    category_base_mask &= portfolio_targets[f"return_lag_{lag}"].notna()
for lag in range(1, 4):
    category_base_mask &= portfolio_targets[f"risk_eligible_lag_{lag}"]

PORTFOLIO_CATEGORY_SAMPLES = {}
for category in RISK_CATEGORIES:
    category_mask = category_base_mask.copy()
    for lag in range(1, 4):
        category_mask &= portfolio_targets[f"risk_{category}_lag_{lag}"].notna()
    category_sample = (
        portfolio_targets.loc[category_mask]
        .sort_values("bar_start_utc")
        .reset_index(drop=True)
    )
    if category_sample.empty:
        raise ValueError(f"No p=3-complete portfolio sample for {category}")
    PORTFOLIO_CATEGORY_SAMPLES[category] = category_sample

# Report the all-category intersection as a comparability diagnostic, but do not discard
# category-specific observations from the primary tests.
intersection_mask = category_base_mask.copy()
for lag in range(1, 4):
    for category in RISK_CATEGORIES:
        intersection_mask &= portfolio_targets[f"risk_{category}_lag_{lag}"].notna()
PORTFOLIO_CATEGORY_INTERSECTION = (
    portfolio_targets.loc[intersection_mask]
    .sort_values("bar_start_utc")
    .reset_index(drop=True)
)


def portfolio_category_support(sample: pd.DataFrame, category: str) -> dict:
    result = {}
    for lag in (1, 2, 3):
        column = f"risk_{category}_lag_{lag}"
        nonzero = sample[column].ne(0)
        result[f"nonzero_rows_lag_{lag}"] = int(nonzero.sum())
        result[f"nonzero_days_lag_{lag}"] = int(sample.loc[nonzero, "trading_date"].nunique())
        result[f"unique_values_lag_{lag}"] = int(sample[column].nunique(dropna=True))
    return result


def category_not_estimable_record(sample, category, q, p, reason):
    support = portfolio_category_support(sample, category)
    return {
        "risk_category": category,
        "risk_lags_p": p,
        "return_lags_q": q,
        "n_obs": int(len(sample)),
        "n_trading_days": int(sample["trading_date"].nunique()),
        "status": "not_estimable",
        "reason": reason,
        "needs_caution": True,
        "risk_sd_full_eligible": CATEGORY_SD[category],
        **support,
        "risk_beta_sum": np.nan,
        "effect_1SD_percent": np.nan,
        "effect_1SD_CI_low_percent": np.nan,
        "effect_1SD_CI_high_percent": np.nan,
        "day_cluster_F": np.nan,
        "day_cluster_df_denom": np.nan,
        "day_cluster_p": np.nan,
        "HAC5_p": np.nan,
        "adjusted_R2": np.nan,
    }


def fit_portfolio_category_model(sample: pd.DataFrame, category: str, q: int, p: int) -> dict:
    risk_columns = [f"risk_{category}_lag_{lag}" for lag in range(1, p + 1)]
    constant_columns = [column for column in risk_columns if sample[column].nunique(dropna=True) < 2]
    if constant_columns:
        return category_not_estimable_record(
            sample, category, q, p,
            "no within-sample variation in: " + ", ".join(constant_columns),
        )

    y = sample["target_return"].astype(float)
    X_restricted = portfolio_design(sample, q=q, category=None, p=0)
    X_unrestricted = portfolio_design(sample, q=q, category=category, p=p)
    if np.linalg.matrix_rank(X_unrestricted.to_numpy()) != X_unrestricted.shape[1]:
        return category_not_estimable_record(sample, category, q, p, "rank-deficient design matrix")
    if len(sample) <= X_unrestricted.shape[1] + 20:
        return category_not_estimable_record(sample, category, q, p, "insufficient residual degrees of freedom")

    restricted = sm.OLS(y, X_restricted, missing="raise").fit()
    unrestricted = sm.OLS(y, X_unrestricted, missing="raise").fit()
    names = list(X_unrestricted.columns)
    risk_positions = [names.index(column) for column in risk_columns]
    restriction = np.zeros((p, len(names)), dtype=float)
    for row_number, position in enumerate(risk_positions):
        restriction[row_number, position] = 1.0

    day_groups = pd.Categorical(sample["trading_date"]).codes
    day_clustered = unrestricted.get_robustcov_results(
        cov_type="cluster",
        groups=day_groups,
        use_correction=True,
        df_correction=True,
    )
    day_wald = day_clustered.wald_test(restriction, use_f=True, scalar=True)
    hac = unrestricted.get_robustcov_results(cov_type="HAC", maxlags=5, use_correction=True)
    hac_wald = hac.wald_test(restriction, use_f=True, scalar=True)

    params = np.asarray(day_clustered.params, dtype=float)
    covariance = np.asarray(day_clustered.cov_params(), dtype=float)
    sum_vector = np.zeros(len(names), dtype=float)
    sum_vector[risk_positions] = 1.0
    beta_sum = float(sum_vector @ params)
    beta_sum_variance = float(sum_vector @ covariance @ sum_vector)
    beta_sum_se = np.sqrt(beta_sum_variance) if beta_sum_variance >= 0 else np.nan
    inference_df = float(day_wald.df_denom)
    critical_value = float(stats.t.ppf(0.975, inference_df))
    ci_low = beta_sum - critical_value * beta_sum_se if np.isfinite(beta_sum_se) else np.nan
    ci_high = beta_sum + critical_value * beta_sum_se if np.isfinite(beta_sum_se) else np.nan
    risk_sd = CATEGORY_SD[category]
    support = portfolio_category_support(sample, category)
    included_nonzero_days = [support[f"nonzero_days_lag_{lag}"] for lag in range(1, p + 1)]

    return {
        "risk_category": category,
        "risk_lags_p": p,
        "return_lags_q": q,
        "n_obs": int(unrestricted.nobs),
        "n_trading_days": int(sample["trading_date"].nunique()),
        "status": "estimated",
        "reason": "",
        "needs_caution": min(included_nonzero_days) < 5,
        "risk_sd_full_eligible": risk_sd,
        **support,
        "risk_beta_sum": beta_sum,
        "effect_1SD_percent": 100 * np.expm1(beta_sum * risk_sd),
        "effect_1SD_CI_low_percent": 100 * np.expm1(ci_low * risk_sd),
        "effect_1SD_CI_high_percent": 100 * np.expm1(ci_high * risk_sd),
        "day_cluster_F": float(np.asarray(day_wald.statistic).squeeze()),
        "day_cluster_df_denom": inference_df,
        "day_cluster_p": float(np.asarray(day_wald.pvalue).squeeze()),
        "HAC5_p": float(np.asarray(hac_wald.pvalue).squeeze()),
        "adjusted_R2": float(unrestricted.rsquared_adj),
    }


portfolio_category_results = pd.DataFrame(
    [
        fit_portfolio_category_model(PORTFOLIO_CATEGORY_SAMPLES[category], category, Q_SELECTED, p)
        for category in RISK_CATEGORIES
        for p in (1, 2, 3)
    ]
)
if len(portfolio_category_results) != 15:
    raise ValueError("Expected five categories x three risk-lag specifications")

portfolio_category_results["BH_q_all_estimable"] = np.nan
portfolio_category_results["Holm_p_all_estimable"] = np.nan
valid = portfolio_category_results["status"].eq("estimated") & portfolio_category_results["day_cluster_p"].notna()
if valid.any():
    portfolio_category_results.loc[valid, "BH_q_all_estimable"] = multipletests(
        portfolio_category_results.loc[valid, "day_cluster_p"], method="fdr_bh"
    )[1]
    portfolio_category_results.loc[valid, "Holm_p_all_estimable"] = multipletests(
        portfolio_category_results.loc[valid, "day_cluster_p"], method="holm"
    )[1]
portfolio_category_results["raw_p_below_0_05"] = valid & portfolio_category_results["day_cluster_p"].lt(0.05)
portfolio_category_results["FDR_5pct_significant"] = valid & portfolio_category_results["BH_q_all_estimable"].lt(0.05)

display(Markdown("### 等权组合：五类风险的全部检验"))
display(
    portfolio_category_results[[
        "risk_category", "risk_lags_p", "return_lags_q", "n_obs", "n_trading_days",
        "status", "reason", "needs_caution",
        "nonzero_rows_lag_1", "nonzero_days_lag_1",
        "nonzero_rows_lag_2", "nonzero_days_lag_2",
        "nonzero_rows_lag_3", "nonzero_days_lag_3",
        "risk_beta_sum", "effect_1SD_percent",
        "effect_1SD_CI_low_percent", "effect_1SD_CI_high_percent",
        "day_cluster_F", "day_cluster_p", "BH_q_all_estimable",
        "Holm_p_all_estimable", "FDR_5pct_significant", "HAC5_p", "adjusted_R2",
    ]].style.format(
        {
            "risk_beta_sum": "{:.8f}",
            "effect_1SD_percent": "{:.5f}",
            "effect_1SD_CI_low_percent": "{:.5f}",
            "effect_1SD_CI_high_percent": "{:.5f}",
            "day_cluster_F": "{:.4f}",
            "day_cluster_p": "{:.6g}",
            "BH_q_all_estimable": "{:.6g}",
            "Holm_p_all_estimable": "{:.6g}",
            "HAC5_p": "{:.6g}",
            "adjusted_R2": "{:.6f}",
        },
        na_rep="—",
    ).hide(axis="index")
)

raw_p_matrix = portfolio_category_results.pivot(index="risk_category", columns="risk_lags_p", values="day_cluster_p")
raw_p_matrix.columns = [f"p={int(value)}" for value in raw_p_matrix.columns]
bh_matrix = portfolio_category_results.pivot(index="risk_category", columns="risk_lags_p", values="BH_q_all_estimable")
bh_matrix.columns = [f"p={int(value)}" for value in bh_matrix.columns]
display(Markdown("### 原始 p 值矩阵"))
display(raw_p_matrix.style.format("{:.4f}", na_rep="不可估计"))
display(Markdown("### 对全部可估计分类检验统一 BH 校正后的 q 值"))
display(bh_matrix.style.format("{:.4f}", na_rep="不可估计"))

sample_variation_rows = []
for category in RISK_CATEGORIES:
    category_sample = PORTFOLIO_CATEGORY_SAMPLES[category]
    values = pd.concat(
        [category_sample[f"risk_{category}_lag_{lag}"] for lag in (1, 2, 3)],
        ignore_index=True,
    )
    sample_variation_rows.append(
        {
            "risk_category": category,
            "sample_rows": len(category_sample),
            "trading_days": int(category_sample["trading_date"].nunique()),
            "nonzero_rows_lag_1": int(category_sample[f"risk_{category}_lag_1"].ne(0).sum()),
            "nonzero_days_lag_1": int(category_sample.loc[category_sample[f"risk_{category}_lag_1"].ne(0), "trading_date"].nunique()),
            "nonzero_rows_lag_2": int(category_sample[f"risk_{category}_lag_2"].ne(0).sum()),
            "nonzero_days_lag_2": int(category_sample.loc[category_sample[f"risk_{category}_lag_2"].ne(0), "trading_date"].nunique()),
            "nonzero_rows_lag_3": int(category_sample[f"risk_{category}_lag_3"].ne(0).sum()),
            "nonzero_days_lag_3": int(category_sample.loc[category_sample[f"risk_{category}_lag_3"].ne(0), "trading_date"].nunique()),
            "lag_cells": len(values),
            "nonzero_lag_cells": int(values.ne(0).sum()),
            "unique_lag_values": int(values.nunique()),
            "lag_value_sd": float(values.std(ddof=1)),
        }
    )
display(Markdown("### 各类别自身共同回归样本内的风险变异"))
display(pd.DataFrame(sample_variation_rows).style.format({"lag_value_sd": "{:.6f}"}).hide(axis="index"))

sample_by_hour = pd.DataFrame(
    {
        category: sample.groupby("target_hour").size()
        for category, sample in PORTFOLIO_CATEGORY_SAMPLES.items()
    }
).fillna(0).astype(int)
display(Markdown("### 各类别 p=3 完整样本的目标时段构成"))
display(sample_by_hour)
display(Markdown(f"五类全部非空的交集样本为 **{len(PORTFOLIO_CATEGORY_INTERSECTION)}** 条。"))

estimated_count = int(valid.sum())
raw_count = int(portfolio_category_results["raw_p_below_0_05"].sum())
fdr_count = int(portfolio_category_results["FDR_5pct_significant"].sum())
display(Markdown("### 机械性结论"))
display(Markdown(
    f"计划15项分类检验，其中{estimated_count}项可估计、{15-estimated_count}项不可估计；"
    f"{raw_count}项原始p值低于0.05，统一BH校正后{fdr_count}项达到5% FDR。"
))

### 等权组合：五类风险的全部检验

risk_category,risk_lags_p,return_lags_q,n_obs,n_trading_days,status,reason,needs_caution,nonzero_rows_lag_1,nonzero_days_lag_1,nonzero_rows_lag_2,nonzero_days_lag_2,nonzero_rows_lag_3,nonzero_days_lag_3,risk_beta_sum,effect_1SD_percent,effect_1SD_CI_low_percent,effect_1SD_CI_high_percent,day_cluster_F,day_cluster_p,BH_q_all_estimable,Holm_p_all_estimable,FDR_5pct_significant,HAC5_p,adjusted_R2
loss_of_human_control,1,3,53,18,estimated,,False,12,7,9,6,5,3,0.00015150,0.01427,-0.02114,0.04970,0.7229,0.407015,0.678359,1,False,0.379188,0.082170
loss_of_human_control,2,3,53,18,estimated,,False,12,7,9,6,5,3,0.00023701,0.02233,-0.01975,0.06443,0.6588,0.530189,0.683566,1,False,0.498172,0.064277
loss_of_human_control,3,3,53,18,estimated,,True,12,7,9,6,5,3,0.00075386,0.07105,-0.02990,0.17209,1.1530,0.356447,0.678359,1,False,0.350193,0.091473
malicious_use,1,3,54,18,estimated,,False,8,6,7,5,8,7,0.00019164,0.02891,-0.03682,0.09468,0.8609,0.366487,0.678359,1,False,0.339359,0.073491
malicious_use,2,3,54,18,estimated,,False,8,6,7,5,8,7,-0.00002916,-0.00440,-0.17580,0.16730,0.4640,0.636468,0.683566,1,False,0.606314,0.059734
malicious_use,3,3,54,18,estimated,,False,8,6,7,5,8,7,0.00003180,0.00480,-0.15508,0.16493,0.5056,0.683566,0.683566,1,False,0.683089,0.037494
job_displacement,1,3,51,18,estimated,,False,5,5,10,8,7,6,-0.00015615,-0.02405,-0.06697,0.01889,1.3962,0.253619,0.678359,1,False,0.294753,0.073493
job_displacement,2,3,51,18,estimated,,False,5,5,10,8,7,6,-0.00029270,-0.04507,-0.11475,0.02465,0.9628,0.401692,0.678359,1,False,0.4807,0.070595
job_displacement,3,3,51,18,estimated,,False,5,5,10,8,7,6,-0.00034984,-0.05387,-0.16406,0.05645,0.6672,0.583689,0.683566,1,False,0.649393,0.047680
discrimination,1,3,54,18,not_estimable,no within-sample variation in: risk_discrimination_lag_1,True,0,0,0,0,0,0,—,—,—,—,—,—,—,—,False,—,—


### 原始 p 值矩阵

,p=1,p=2,p=3
risk_category,,,
discrimination,不可估计,不可估计,不可估计
job_displacement,0.2536,0.4017,0.5837
loss_of_human_control,0.4070,0.5302,0.3564
malicious_use,0.3665,0.6365,0.6836
privacy_violation,0.3381,不可估计,不可估计


### 对全部可估计分类检验统一 BH 校正后的 q 值

...

### 各类别 p=3 完整样本的目标时段构成

,loss_of_human_control,malicious_use,job_displacement,discrimination,privacy_violation
target_hour,,,,,
13:00,18,18,17,18,17
14:00,18,18,17,18,17
15:00,17,18,17,18,17


五类全部非空的交集样本为 **48** 条。

### 机械性结论

计划15项分类检验，其中10项可估计、5项不可估计；0项原始p值低于0.05，统一BH校正后0项达到5% FDR。

## 5. Interpretation limits

1. These are five separate category models. Because one document can contribute to multiple categories, a category coefficient is not an independently identified causal effect holding the other categories fixed.
2. A structurally constant category in the regression sample cannot be tested. Its missing p-value means “not estimable,” not “no effect.”
3. `needs_caution=True` means at least one included lag is supported by fewer than five nonzero trading days. Such a p-value is especially fragile even when the model is technically estimable.
4. For p>1, the +1 SD cumulative effect means every included lag rises by one category-specific full-period SD together; it is not a one-hour impulse response.
5. The fixed 20-stock composition is retained. No bridge-hour return is filled and the portfolio is never silently reconstituted.
6. The X News collection has zero strictly complete-coverage hours and all queries hit their result cap. Results describe only the observed X News sample.
7. Granger predictability is temporal predictive content, not structural causality.